<a href="https://colab.research.google.com/github/kimjiwoo2/Pill-agent/blob/develop/src/demo/06_jw_e2e_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## [준비 파트] — 발표 전 미리 1회만. 데모 중엔 실행하지 않음.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# [셀 A] 설치 (numpy<2 : paddle/torch 호환)
# 이 셀 실행 후 반드시 런타임 재시작 (셀 B). 재시작 후엔 이 셀 건너뜀.
!pip -q install "numpy<2" paddlepaddle==3.1.0 "paddleocr>=3.0.0,<3.8" \
                pymysql sqlalchemy ultralytics

In [ ]:
# [셀 B] 런타임 재시작
# 위 설치가 numpy 버전을 바꾸므로 재시작 필요. 아래 한 줄로 자동 재시작:
import os; os.kill(os.getpid(), 9)
#
# ※ 재시작되면 셀 A·B 는 건너뛰고 [셀 1] 부터 실행.
#   (Colab 세션이 살아있는 동안은 재설치 불필요)

## [데모 파트] — 발표장에서 여기부터. 설치·재시작 없음.

In [ ]:
# [셀 1] 환경 확인
import numpy as np, cv2, torch
assert np.__version__.startswith("1."), "numpy 2.x — 준비 파트 재시작 안 됨. 셀 A·B 확인"
print(f"numpy {np.__version__} | cv2 {cv2.__version__} | torch {torch.__version__} | CUDA {torch.cuda.is_available()}")

In [ ]:
# [셀 2] 레포 + 모듈 import
import os
if not os.path.exists('/content/Pill-agent'):
    os.system('git clone -b develop https://github.com/kimjiwoo2/Pill-agent.git')
os.system('cd /content/Pill-agent && git pull origin develop')

import sys
sys.path.append('/content/Pill-agent/src/matching')
from pill_e2e import ScenePipeline
from pill_fusion import load_drug_master
print("import OK")

In [ ]:
# [셀 3] 경로 설정 (CONFIG)
BASE = '/content/drive/MyDrive/ToBigs/2425/Pillot/'
JW   = f'{BASE}jiwoo/20k/'

CFG = dict(
    ckpt     = f'{JW}checkpoints/best_20k_v3_5_nosampler_ep16_v3.pth',
    ts       = f'{JW}checkpoints/temperature_20k_v3_5_nosampler_ep16_v3.pkl',
    encoders = f'{JW}data/label_encoders_20k_11cls.pkl',
    weights  = '/content/Pill-agent/src/matching/fw_final.json',
    yolo_pt  = f'{BASE}juhyeong/detect_v1_results/runs/yolo11n_detect_v1/weights/best.pt',
)

In [ ]:
# [셀 4] DB 로드 (1회)
from google.colab import userdata

class _Args: pass
_a = _Args()
_a.encoders        = CFG['encoders']
_a.drug_master_csv = None                # None → MySQL 사용
_a.db_user = 'jiwoo_admin'
_a.db_pw   = userdata.get('PW')
_a.db_host = '103.218.161.72'
_a.db_name = 'pilliot_db'
db = load_drug_master(_a)

In [ ]:
# [셀 5] 파이프라인 조립 (1회)
pipe = ScenePipeline(
    ckpt=CFG['ckpt'], ts_path=CFG['ts'], encoders=CFG['encoders'],
    weights_json=CFG['weights'], yolo_pt=CFG['yolo_pt'], db=db,
    ocr_mode='accurate',                     # 데모 속도 문제면 'fast'
)
print("파이프라인 준비 완료")

In [ ]:
# [셀 6] 데모 이미지 준비 (원본 씬 zip 풀기)
#  데모용 큐레이션 이미지로 변경하며 테스트
import zipfile, glob
RAW_ZIP = f'{BASE}dataset/pilliot_test_set/test_raw_images.zip'
if not os.path.exists('/content/raw_scenes'):
    zipfile.ZipFile(RAW_ZIP).extractall('/content/raw_scenes')
scenes = sorted(glob.glob('/content/raw_scenes/**/*.png', recursive=True) +
                glob.glob('/content/raw_scenes/**/*.jpg', recursive=True))
print(f"원본 씬 {len(scenes)}장 준비")

In [ ]:
# [셀 7] 단일 씬 데모 --> 웹 데모에서는 사용 안 함, 웹용으로 변경
import matplotlib.pyplot as plt

def demo(scene_path, k=3):
    scene = cv2.imread(scene_path)
    assert scene is not None, f"이미지 로드 실패: {scene_path}"
    results = pipe.run_scene(scene, k=k)

    print(f"검출된 알약: {len(results)}개\n")
    for r in results:
        print(f"── 알약 #{r['pill_idx']} (검출 {r['det_conf']:.2f}) "
              f"OCR='{r['ocr_raw']}' ({r['ocr_conf']:.2f})")
        for t in r['topk']:
            print(f"     {t['item_seq']}  {t['score']:+.2f}  {t['name']}")
        print()

    vis = pipe.visualize(scene, results)
    plt.figure(figsize=(12, 9)); plt.imshow(vis); plt.axis('off')
    plt.title(os.path.basename(scene_path)); plt.show()
    return results

# 실행 예:
# results = demo(scenes[0], k=3)

In [ ]:
# [셀 8] (선택) 캐시 생성 — 데모 지연 제거
#   큐레이션한 씬들을 미리 처리해서 저장. 발표장에선 캐시만 로드 → 즉시 표시.
def build_cache(scene_paths, out='/content/drive/MyDrive/demo_cache.pkl', k=3):
    import pickle
    cache = {}
    for p in scene_paths:
        scene = cv2.imread(p)
        cache[p] = pipe.run_scene(scene, k=k)
        print(f"  캐시: {os.path.basename(p)} ({len(cache[p])}개 알약)")
    with open(out, 'wb') as f:
        pickle.dump(cache, f)
    print(f"캐시 저장 → {out}")

def demo_cached(scene_path, cache_file='/content/drive/MyDrive/demo_cache.pkl'):
    """캐시된 결과 즉시 표시 (모델 로드 불필요 → 재시작·GPU충돌 걱정 없음)."""
    import pickle
    with open(cache_file, 'rb') as f:
        cache = pickle.load(f)
    results = cache[scene_path]
    scene = cv2.imread(scene_path)
    for r in results:
        print(f"알약 #{r['pill_idx']} OCR='{r['ocr_raw']}' → {r['topk'][0]['name']}")
    vis = pipe.visualize(scene, results)
    plt.figure(figsize=(12, 9)); plt.imshow(vis); plt.axis('off'); plt.show()
    return results

# 데모 전 준비:  build_cache([scenes[0], scenes[3], scenes[7]])    # 큐레이션한 것만
# 발표장에서:    demo_cached(scenes[0])                            # 즉시